In [2]:
import sys
from pathlib import Path

current_dir = Path.cwd()

def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / 'pyproject.toml').exists() and (p / 'bcosgnn').is_dir():
            return p
    raise RuntimeError('Could not locate repo root (pyproject.toml + bcosgnn/).')

project_root = find_repo_root(current_dir)

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

print(f"Repo root added: {project_root}")

Repo root added: /Users/shaique/Desktop/BioInf_IMP/NMM_group/ICML/bcos_gnn/bcosgnn


In [3]:
import random
from collections import defaultdict
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import networkx as nx
from sklearn.metrics import roc_auc_score
from torch_geometric.loader import DataLoader
from torch_geometric.utils import to_networkx
from torch_geometric.nn.conv import GINEConv
from torch_geometric.nn.aggr import MeanAggregation, SumAggregation

from tqdm.auto import tqdm

from bcos.modules import BcosLinear
from bcosgnn.explain_edge_attr import explain as explain_edge_attr
import os
import shutil
from torch_geometric.data import InMemoryDataset
import tqdm
import sys
import os
# Add the project root to the Python path
project_root = '/home/moso00002/Desktop/gnn/bcosgnn-bcos_gnn_shaique'
if project_root not in sys.path:
    sys.path.append(project_root)

import functools
import itertools
import operator
from typing import Any
import torch
from torch_geometric.data import Dataset, download_url
from torch.utils.data import random_split
import numpy as np
import polars as pl
import seaborn as sns
import matplotlib.pyplot as plt
import torch
from bcos.modules import BcosLinear, BcosSequential
from sklearn.model_selection import train_test_split
from torch.nn import BCEWithLogitsLoss
from torch_geometric.datasets import TUDataset
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import MessagePassing
from torch_geometric.nn.aggr import SumAggregation
from torch_geometric.utils import add_self_loops, degree
from torchmetrics import AUROC
from torchmetrics.classification import BinaryAccuracy
from tqdm import tqdm
import networkx as nx
import torch.nn.functional as F
from bcosgnn.explain import explain
from bcosgnn.evaluation import get_attribution_scores

/Users/shaique/Desktop/BioInf_IMP/NMM_group/ICML/bcos_gnn/bcosgnn/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing, global_add_pool

# -------------------------
# Vanilla GIN Convolution (Maskable for GSAT)
# -------------------------
class MaskableGINConv(MessagePassing):
    def __init__(self, mlp: nn.Module, eps=0.0, train_eps=False):
        super().__init__(aggr="add")
        self.mlp = mlp
        if train_eps:
            self.eps = nn.Parameter(torch.tensor([eps], dtype=torch.float))
        else:
            self.register_buffer("eps", torch.tensor([eps], dtype=torch.float))

    def forward(self, x, edge_index, edge_weight=None):
        # Propagate messages; pass edge_weight to the message function
        out = self.propagate(edge_index, x=x, edge_weight=edge_weight)
        
        # Add scaled self-loops
        out = (1 + self.eps) * x + out
        
        # Apply the GIN MLP
        return self.mlp(out)

    def message(self, x_j, edge_weight=None):
        # Scale the messages by the GSAT attention mask
        msg = x_j
        if edge_weight is not None:
            msg = msg * edge_weight.view(-1, 1)
        return msg


# -------------------------
# Vanilla GIN Backbone
# -------------------------
class GINBackbone(nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int, num_classes: int, drop_ratio=0.5):
        super().__init__()
        
        # Initial node feature projection
        self.node_proj = nn.Linear(in_dim, hidden_dim)
        
        # Standard GIN uses a 2-layer MLP inside the convolution
        mlp1 = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
        self.conv1 = MaskableGINConv(mlp1, train_eps=True)
        
        mlp2 = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
        self.conv2 = MaskableGINConv(mlp2, train_eps=True)
        
        self.drop_ratio = drop_ratio
        
        # Final Readout Classifier
        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, x, edge_index, edge_weight=None, batch=None):
        if batch is None:
            batch = torch.zeros(x.size(0), dtype=torch.long, device=x.device)
            
        x = self.node_proj(x.float())
        
        # Layer 1
        x = self.conv1(x, edge_index, edge_weight=edge_weight)
        x = F.relu(x)
        
        # Layer 2
        x = self.conv2(x, edge_index, edge_weight=edge_weight)
        x = F.relu(x)
        
        # Graph-level pooling
        hg = global_add_pool(x, batch)
        
        # Dropout & Classify
        hg = F.dropout(hg, p=self.drop_ratio, training=self.training)
        return self.classifier(hg)

In [5]:
# -------------------------
# GSAT Loss + Seed Helper
# -------------------------
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def gsat_loss(pred_logits, ground_truth_labels, mask_logits, r=0.7, pred_loss_coef=1.0, info_loss_coef=1.0):
    criterion = nn.CrossEntropyLoss()
    pred_loss = criterion(pred_logits, ground_truth_labels)

    mask_probs = torch.sigmoid(mask_logits)
    prior_target = torch.full_like(mask_probs, 1.0 - r)
    info_loss = F.binary_cross_entropy(mask_probs, prior_target, reduction="mean")

    loss = (pred_loss_coef * pred_loss) + (info_loss_coef * info_loss)
    return loss, pred_loss, info_loss

In [6]:
# -------------------------
# GSAT Wrapper (No Edge Attributes)
# -------------------------
class GSAT(nn.Module):
    def __init__(self, backbone, in_channels, hidden_channels, temperature=1.0):
        super().__init__()
        self.backbone = backbone
        self.temperature = temperature

        # Attention MLP: uses concatenated source/target node features
        self.att_mlp = nn.Sequential(
            nn.Linear(in_channels * 2, hidden_channels),
            nn.ReLU(),
            nn.Linear(hidden_channels, hidden_channels),
            nn.ReLU(),
            nn.Linear(hidden_channels, 1),
        )

        for m in self.att_mlp.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)

    def get_mask(self, x, edge_index, training=True):
        row, col = edge_index
        edge_rep = torch.cat([x[row], x[col]], dim=-1)
        edge_logits = self.att_mlp(edge_rep).view(-1)

        if training:
            u = torch.rand_like(edge_logits)
            noise = torch.log(u + 1e-8) - torch.log(1 - u + 1e-8)
            mask = torch.sigmoid((edge_logits + noise) / self.temperature)
        else:
            mask = torch.sigmoid(edge_logits)

        return mask, edge_logits

    def forward(self, data, training=True):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        mask, mask_logits = self.get_mask(x, edge_index, training=training)
        pred_logits = self.backbone(x, edge_index, edge_weight=mask, batch=batch)
        return pred_logits, mask, mask_logits

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch_geometric.datasets import TUDataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import MessagePassing, global_add_pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score
import torch.optim as optim

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


In [9]:
# Load NCI1
dataset = TUDataset(root='data/TUDataset', name='NCI1')

# Extract necessary dimensions for the models
in_dim = dataset.num_features
num_classes = dataset.num_classes

print(f"Dataset: {dataset.name} | Graphs: {len(dataset)}")
print(f"Node Features (in_dim): {in_dim} | Classes (num_classes): {num_classes}")

Dataset: NCI1 | Graphs: 4110
Node Features (in_dim): 37 | Classes (num_classes): 2


In [10]:
class MaskableGINConv(MessagePassing):
    def __init__(self, mlp: nn.Module, eps=0.0, train_eps=False):
        super().__init__(aggr="add")
        self.mlp = mlp
        if train_eps:
            self.eps = nn.Parameter(torch.tensor([eps], dtype=torch.float))
        else:
            self.register_buffer("eps", torch.tensor([eps], dtype=torch.float))

    def forward(self, x, edge_index, edge_weight=None):
        out = self.propagate(edge_index, x=x, edge_weight=edge_weight)
        out = (1 + self.eps) * x + out
        return self.mlp(out)

    def message(self, x_j, edge_weight=None):
        msg = x_j
        if edge_weight is not None:
            msg = msg * edge_weight.view(-1, 1)
        return msg

class GINBackbone(nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int, num_classes: int, drop_ratio=0.5):
        super().__init__()
        self.node_proj = nn.Linear(in_dim, hidden_dim)
        
        mlp1 = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.BatchNorm1d(hidden_dim),
            nn.ReLU(), nn.Linear(hidden_dim, hidden_dim)
        )
        self.conv1 = MaskableGINConv(mlp1, train_eps=True)
        
        mlp2 = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.BatchNorm1d(hidden_dim),
            nn.ReLU(), nn.Linear(hidden_dim, hidden_dim)
        )
        self.conv2 = MaskableGINConv(mlp2, train_eps=True)
        
        self.drop_ratio = drop_ratio
        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, x, edge_index, edge_weight=None, batch=None):
        if batch is None:
            batch = torch.zeros(x.size(0), dtype=torch.long, device=x.device)
            
        x = self.node_proj(x.float())
        x = F.relu(self.conv1(x, edge_index, edge_weight=edge_weight))
        x = F.relu(self.conv2(x, edge_index, edge_weight=edge_weight))
        
        hg = global_add_pool(x, batch)
        hg = F.dropout(hg, p=self.drop_ratio, training=self.training)
        return self.classifier(hg)

class GSAT(nn.Module):
    def __init__(self, backbone, in_channels, hidden_channels, temperature=1.0):
        super().__init__()
        self.backbone = backbone
        self.temperature = temperature
        self.att_mlp = nn.Sequential(
            nn.Linear(in_channels * 2, hidden_channels), nn.ReLU(),
            nn.Linear(hidden_channels, hidden_channels), nn.ReLU(),
            nn.Linear(hidden_channels, 1),
        )
        for m in self.att_mlp.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)

    def get_mask(self, x, edge_index, training=True):
        row, col = edge_index
        edge_rep = torch.cat([x[row], x[col]], dim=-1)
        edge_logits = self.att_mlp(edge_rep).view(-1)
        if training:
            u = torch.rand_like(edge_logits)
            noise = torch.log(u + 1e-8) - torch.log(1 - u + 1e-8)
            mask = torch.sigmoid((edge_logits + noise) / self.temperature)
        else:
            mask = torch.sigmoid(edge_logits)
        return mask, edge_logits

    def forward(self, data, training=True):
        x, edge_index, batch = data.x.float(), data.edge_index, data.batch
        mask, mask_logits = self.get_mask(x, edge_index, training=training)
        pred_logits = self.backbone(x, edge_index, edge_weight=mask, batch=batch)
        return pred_logits, mask, mask_logits

In [11]:
class EarlyStopping:
    def __init__(self, patience=25, min_delta=0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_score = None
        self.early_stop = False

    def __call__(self, val_acc):
        if self.best_score is None:
            self.best_score = val_acc
        elif val_acc < self.best_score + self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = val_acc
            self.counter = 0

def train(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    for data in loader:
        data = data.to(device)
        optimizer.zero_grad()
        
        # GSAT returns a tuple: (pred_logits, mask, mask_logits)
        out, mask, mask_logits = model(data, training=True)
        
        loss = criterion(out, data.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * data.num_graphs
    return total_loss / len(loader.dataset)

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    y_true, y_pred = [], []
    for data in loader:
        data = data.to(device)
        
        # Disable training mode to freeze Gumbel-Softmax noise and Dropout
        out, mask, mask_logits = model(data, training=False)
        
        pred = out.argmax(dim=1)
        y_true.extend(data.y.cpu().numpy())
        y_pred.extend(pred.cpu().numpy())
    
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='macro')
    return acc, f1

In [12]:
seeds = [42, 123, 999, 7, 2024]

# Hyperparameters
HIDDEN_DIM = 64
NUM_LAYERS = 5
DROPOUT = 0.5
LR = 1e-3
EPOCHS = 200
BATCH_SIZE = 128
PATIENCE = 25

results = {"acc": [], "f1": []}

print(f"\n--- Starting 5-Seed Run (Early Stopping & Scheduler) ---")

for seed in seeds:
    # Reproducibility
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    
    # Stratified Split (80/10/10)
    all_indices = np.arange(len(dataset))
    labels = [d.y.item() for d in dataset]
    
    train_idx, temp_idx, _, temp_labels = train_test_split(
        all_indices, labels, test_size=0.2, stratify=labels, random_state=seed
    )
    val_idx, test_idx = train_test_split(
        temp_idx, test_size=0.5, stratify=temp_labels, random_state=seed
    )
    
    train_loader = DataLoader(dataset[torch.tensor(train_idx)], batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(dataset[torch.tensor(val_idx)], batch_size=BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(dataset[torch.tensor(test_idx)], batch_size=BATCH_SIZE, shuffle=False)
    
    # Model Setup
    backbone = GINBackbone(
        in_dim=in_dim,
        hidden_dim=HIDDEN_DIM,
        num_classes=num_classes,
        drop_ratio=DROPOUT
    )
    
    model = GSAT(
        backbone=backbone,
        in_channels=in_dim,
        hidden_channels=HIDDEN_DIM,
        temperature=1.0,
    ).to(device)
    
    optimizer = optim.Adam(model.parameters(), lr=LR)
    criterion = nn.CrossEntropyLoss()
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=PATIENCE, min_lr=1e-6
    )
    early_stopper = EarlyStopping(patience=PATIENCE)
    
    best_val_acc = 0.0
    final_test_acc = 0.0
    final_test_f1 = 0.0
    
    for epoch in range(1, EPOCHS + 1):
        loss = train(model, train_loader, optimizer, criterion)
        val_acc, val_f1 = evaluate(model, val_loader)
        
        scheduler.step(val_acc)
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            final_test_acc, final_test_f1 = evaluate(model, test_loader)
            
        early_stopper(val_acc)
        if early_stopper.early_stop:
            print(f"Seed {seed}: Early stopping triggered at epoch {epoch}")
            break

    results["acc"].append(final_test_acc)
    results["f1"].append(final_test_f1)
    
    print(f"Seed {seed}: Best Val {best_val_acc:.4f} | Test Acc {final_test_acc:.4f} | Test F1 {final_test_f1:.4f}")

print(f"\n=== NCI1 Final Results (5 Seeds) ===")
print(f"Mean Accuracy: {np.mean(results['acc']):.4f} ± {np.std(results['acc']):.4f}")
print(f"Mean F1 Score: {np.mean(results['f1']):.4f} ± {np.std(results['f1']):.4f}")


--- Starting 5-Seed Run (Early Stopping & Scheduler) ---
Seed 42: Early stopping triggered at epoch 161
Seed 42: Best Val 0.8248 | Test Acc 0.7883 | Test F1 0.7879
Seed 123: Early stopping triggered at epoch 181
Seed 123: Best Val 0.8297 | Test Acc 0.8029 | Test F1 0.8025
Seed 999: Early stopping triggered at epoch 71
Seed 999: Best Val 0.7737 | Test Acc 0.7640 | Test F1 0.7637
Seed 7: Early stopping triggered at epoch 117
Seed 7: Best Val 0.7956 | Test Acc 0.8151 | Test F1 0.8148
Seed 2024: Early stopping triggered at epoch 84
Seed 2024: Best Val 0.7640 | Test Acc 0.7494 | Test F1 0.7489

=== NCI1 Final Results (5 Seeds) ===
Mean Accuracy: 0.7839 ± 0.0243
Mean F1 Score: 0.7836 ± 0.0243
